# Run One Model With The Python API

This notebook creates a tiny synthetic JSONL dataset, trains `AutoSTPP` and
`PoissonGMM` for one CPU epoch, evaluates them, samples next events, and makes a
small predictive scatter plot.

It does not require Hugging Face data or a GPU. In Colab, run it from the root
of a cloned Seahorse repository, then execute the install cell.


## Install/import

The local repository install is only needed when `unified_stpp` is not already
importable. The cell is safe to run from the repository root.


In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

if Path.cwd().name == "notebooks" and Path.cwd().parent.joinpath("pyproject.toml").exists():
    os.chdir(Path.cwd().parent)

if importlib.util.find_spec("unified_stpp") is None:
    if Path("pyproject.toml").exists():
        %pip install -e .
    else:
        raise RuntimeError("Run this notebook from the Seahorse repository root.")

import matplotlib.pyplot as plt
import numpy as np

from unified_stpp import AutoSTPP, PoissonGMM, load_jsonl


## Generate tiny local JSONL data

The data format is the same one used by real runs: one JSON object per sequence,
with aligned `times` and `locations` arrays.


In [ ]:
import json
from pathlib import Path

import numpy as np


def make_toy_sequences(n_sequences: int, *, seed: int, n_events: int = 10):
    rng = np.random.default_rng(seed)
    records = []
    for seq_id in range(n_sequences):
        gaps = rng.uniform(0.08, 0.22, size=n_events)
        times = np.cumsum(gaps)
        phase = np.linspace(0.0, 1.0, n_events) + 0.11 * seq_id
        locations = np.column_stack(
            [
                0.5 + 0.25 * np.sin(2 * np.pi * phase) + rng.normal(0, 0.01, n_events),
                0.5 + 0.25 * np.cos(2 * np.pi * phase) + rng.normal(0, 0.01, n_events),
            ]
        )
        records.append(
            {
                "times": np.round(times, 4).tolist(),
                "locations": np.round(np.clip(locations, 0.05, 0.95), 4).tolist(),
            }
        )
    return records


def write_jsonl(path: Path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

DATA_DIR = Path("runs/tutorials/01_python_api_data")
write_jsonl(DATA_DIR / "train.jsonl", make_toy_sequences(4, seed=1))
write_jsonl(DATA_DIR / "val.jsonl", make_toy_sequences(2, seed=2))
write_jsonl(DATA_DIR / "test.jsonl", make_toy_sequences(2, seed=3))

print(DATA_DIR)
print((DATA_DIR / "train.jsonl").read_text().splitlines()[0])


## Load data


In [ ]:
train = load_jsonl(DATA_DIR / "train.jsonl")
val = load_jsonl(DATA_DIR / "val.jsonl")
test = load_jsonl(DATA_DIR / "test.jsonl")

print({"train": len(train), "val": len(val), "test": len(test)})
print(test[0].keys())


## Instantiate models

The overrides keep AutoSTPP tiny and reduce its sliding-window length so the
notebook finishes quickly on CPU.


In [ ]:
MAX_EPOCHS = 1
BATCH_SIZE = 2

common_overrides = {
    "data": {"num_workers": 0},
}

auto_overrides = {
    "data": {
        "num_workers": 0,
        "adapter_kwargs": {
            "training_view": "sliding_window",
            "lookback": 3,
            "lookahead": 1,
        },
    },
    "model": {
        "hidden_dim": 8,
        "decoder": {
            "lookback": 3,
            "lookahead": 1,
            "max_history": 3,
            "n_prodnet": 1,
            "hidden_size": 8,
            "num_layers": 1,
            "temporal_mc_samples": 2,
        },
    },
}

model = AutoSTPP(device="cpu", seed=42, config_overrides=auto_overrides)
baseline = PoissonGMM(device="cpu", seed=42, config_overrides=common_overrides)


## Fit


In [ ]:
%%capture fit_output
model.fit(train, val, test, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, dataset_id="toy_python_api")
baseline.fit(train, val, test, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, dataset_id="toy_python_api")


In [ ]:
print("AutoSTPP preset:", model.preset)
print("PoissonGMM preset:", baseline.preset)
print("Training completed.")


## Evaluate

The Python API reports likelihood-oriented metrics for the fitted estimator.


In [ ]:
model_scores = model.evaluate(test)
baseline_scores = baseline.evaluate(test)

print("AutoSTPP", model_scores)
print("PoissonGMM", baseline_scores)


## Predict/sample next events

`predict_next` returns arrays over held-out next-event contexts, not necessarily
one row per input sequence.


In [ ]:
samples = model.predict_next(test, n_samples=16)

print(samples.keys())
print("next_times shape:", samples["next_times"].shape)
print("next_locations shape:", samples["next_locations"].shape)
print("sampling backend:", samples["sampling_backend"])


## Simple visualization

This plot uses the sampled locations returned by `predict_next` directly. It is
not a benchmark figure; it is a quick sanity check for one model.


In [ ]:
first_context = 0
sample_locs = samples["next_locations"][first_context]
true_loc = samples["true_next_locations"][first_context]

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(sample_locs[:, 0], sample_locs[:, 1], alpha=0.7, label="sampled next locations")
ax.scatter([true_loc[0]], [true_loc[1]], marker="x", s=80, label="true next location")
ax.set_title("AutoSTPP next-location samples")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="best")
plt.show()


## What to try next

- Increase `MAX_EPOCHS` for a real experiment.
- Use your own `train.jsonl`, `val.jsonl`, and `test.jsonl` files.
- Use the CLI benchmark notebook when comparing multiple presets.
